In [3]:
import sqlite3
import pandas as pd

In [4]:
df_clean = pd.read_csv("../data/healthcare_indicators_clean.csv")
df_clean.head()

,date,medical_cpi,personal_consumption,disposable_income,unemployment,retail_sales
0,1947-01-01,13.2,NaN,NaN,NaN,NaN
1,1947-02-01,13.3,NaN,NaN,NaN,NaN
2,1947-03-01,13.3,NaN,NaN,NaN,NaN
3,1947-04-01,13.4,NaN,NaN,NaN,NaN
4,1947-05-01,13.5,NaN,NaN,NaN,NaN


In [5]:
conn = sqlite3.connect("../data/healthcare_analysis.db")
df_clean.to_sql("healthcare_indicators", conn, if_exists="replace", index=False)
conn.close()
print("Database created.")

Database created.


In [6]:
query = """
SELECT date, medical_cpi, unemployment
FROM healthcare_indicators
LIMIT 10;
"""
result = pd.read_sql(query, sqlite3.connect("../data/healthcare_analysis.db"))
result

,date,medical_cpi,unemployment
0,1947-01-01,13.2,None
1,1947-02-01,13.3,None
2,1947-03-01,13.3,None
3,1947-04-01,13.4,None
4,1947-05-01,13.5,None
5,1947-06-01,13.5,None
6,1947-07-01,13.5,None
7,1947-08-01,13.6,None
8,1947-09-01,13.7,None
9,1947-10-01,13.8,None


In [7]:
query_yoy = """
SELECT
    date,
    medical_cpi,
    LAG(medical_cpi, 12) OVER (ORDER BY date) AS medical_cpi_last_year,
    ROUND(
        (medical_cpi - LAG(medical_cpi, 12) OVER (ORDER BY date)) 
        * 100.0 / LAG(medical_cpi, 12) OVER (ORDER BY date), 2
    ) AS yoy_pct_change
FROM healthcare_indicators
ORDER BY date;
"""
yoy_result = pd.read_sql(query_yoy, sqlite3.connect("../data/healthcare_analysis.db"))
yoy_result.tail(15)

,date,medical_cpi,medical_cpi_last_year,yoy_pct_change
939,2025-04-01,576.548,561.275,2.72
940,2025-05-01,578.073,564.191,2.46
941,2025-06-01,580.527,564.997,2.75
942,2025-07-01,584.450,564.593,3.52
943,2025-08-01,583.809,564.183,3.48
944,2025-09-01,584.872,566.256,3.29
945,2025-10-01,NaN,567.974,NaN
946,2025-11-01,585.987,569.476,2.90
947,2025-12-01,588.092,570.110,3.15
948,2026-01-01,589.610,571.366,3.19


In [9]:
query_bucket_v2 = """
WITH yoy AS (
    SELECT
        date,
        unemployment,
        medical_cpi,
        (medical_cpi - LAG(medical_cpi, 12) OVER (ORDER BY date)) 
        * 100.0 / LAG(medical_cpi, 12) OVER (ORDER BY date) AS yoy_pct_change
    FROM healthcare_indicators
)
SELECT
    CASE
        WHEN unemployment >= 6.0 THEN 'high_unemployment'
        ELSE 'low_unemployment'
    END AS unemployment_bucket,
    ROUND(AVG(yoy_pct_change), 2) AS avg_yoy_pct_change,
    COUNT(*) AS num_months
FROM yoy
WHERE unemployment IS NOT NULL
  AND yoy_pct_change IS NOT NULL
GROUP BY unemployment_bucket;
"""
bucket_result_v2 = pd.read_sql(query_bucket_v2, sqlite3.connect("../data/healthcare_analysis.db"))
bucket_result_v2

,unemployment_bucket,avg_yoy_pct_change,num_months
0,high_unemployment,6.38,330
1,low_unemployment,4.19,610


In [10]:
yoy_result.to_csv("../data/yoy_medical_cpi.csv", index=False)
bucket_result_v2.to_csv("../data/unemployment_bucket_comparison.csv", index=False)